In [1]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment, NamedStyle
from openpyxl.formatting.rule import CellIsRule, FormulaRule
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.utils import get_column_letter
from datetime import datetime
import os

# Create a new workbook
wb = Workbook()

# Remove default sheet
default_sheet = wb.active
wb.remove(default_sheet)

# Create sheets
dashboard_ws = wb.create_sheet(title="Dashboard")
data_ws = wb.create_sheet(title="Data")
calculations_ws = wb.create_sheet(title="Calculations")

# Hide calculations sheet
calculations_ws.sheet_state = 'hidden'

# ==================== DATA SHEET ====================
# Set column widths for Data sheet
data_ws.column_dimensions['A'].width = 30
data_ws.column_dimensions['B'].width = 15
data_ws.column_dimensions['C'].width = 15
data_ws.column_dimensions['D'].width = 15
data_ws.column_dimensions['E'].width = 15
data_ws.column_dimensions['F'].width = 15

# Headers for Data sheet
data_headers = ["Metric", "Current Value", "Previous Value", "Change", "Target", "Status"]
data_ws.append(data_headers)

# Data rows
data_rows = [
    ["Overall Satisfaction", 0.74, 0.69, "", 0.75, "Below Target"],
    ["Onboarding Experience Score", 68, 65, "", 75, "Below Target"],
    ["Support Response Time (hrs)", 8, 7.5, "", 6, "Below Target"],
    ["Self-Service Adoption %", 35, 32, "", 45, "Below Target"],
    ["Feedback Loop Completion %", 92, 90, "", 90, "On Target"],
    ["NPS Follow-up Rate %", 88, 85, "", 85, "On Target"],
    ["Pain Points Identified", 18, 15, "", "", ""],
    ["Journey Mapping Workshop %", 87, "", "", 100, "In Progress"],
    ["Voice of Customer Program %", 30, "", "", 100, "In Progress"],
    ["Touchpoint Optimization", "", "", "", "", "Late"],
    ["Friction Points Analysis", "", "", "", "", "Not Started"],
    ["CX Performance Progress %", 47, 40, "", 100, "On Track"],
    ["Weeks Remaining", 12, "", "", "", ""],
    # Additional Support Metrics
    ["Support CSAT %", 85, 82, "", 90, "Below Target"],
    ["First Response Time (min)", 15, 18, "", 10, "Below Target"],
    ["Resolution Rate %", 92, 90, "", 95, "Below Target"],
    ["Escalation Rate %", 8, 10, "", 5, "Below Target"],
    ["Self-Service Deflection %", 25, 22, "", 35, "Below Target"],
    ["Customer Effort Score (1-7)", 2.5, 2.8, "", 2.0, "Below Target"],
    ["Sentiment Score (0-100)", 78, 75, "", 85, "Below Target"]
]

for row in data_rows:
    data_ws.append(row)

# Add formulas for Change column (row 2 to 6 for percentages, row 7 for count)
for row in range(2, 7):  # Rows 2-6
    data_ws[f'D{row}'] = f'=B{row}-C{row}'

# Pain points change
data_ws['D7'] = '=B7-C7'

# ==================== CALCULATIONS SHEET ====================
calculations_ws['A1'] = "CALCULATION SHEET - DO NOT EDIT"
calculations_ws['A2'] = "Automatic Calculations"
calculations_ws['B2'] = "Formulas"

# Add some useful formulas
calculations_ws['A3'] = "Satisfaction % Display"
calculations_ws['B3'] = '=TEXT(Data!B2,"0%")'

calculations_ws['A4'] = "Satisfaction Change Display"
calculations_ws['B4'] = '=TEXT(Data!D2,"+0%;-0%;0%")'

calculations_ws['A5'] = "Pain Points Change"
calculations_ws['B5'] = '=IF(Data!D7>0,"+"&Data!D7,Data!D7)'

calculations_ws['A6'] = "Progress Bar"
calculations_ws['B6'] = '=REPT("█",ROUND(Data!B12/10,0))'

# ==================== DASHBOARD SHEET ====================
# Set column widths
for col in range(1, 9):
    dashboard_ws.column_dimensions[get_column_letter(col)].width = 12

# Merge cells for header
dashboard_ws.merge_cells('A1:H1')
dashboard_ws['A1'] = "CUSTOMER EXPERIENCE DASHBOARD"
dashboard_ws['A1'].font = Font(size=16, bold=True, color="2E74B5")
dashboard_ws['A1'].alignment = Alignment(horizontal='center')

# GENERAL Section
dashboard_ws['A3'] = "GENERAL"
dashboard_ws['A3'].font = Font(size=12, bold=True)

# Overview
dashboard_ws['A5'] = "Overview"
dashboard_ws['A5'].font = Font(bold=True)

dashboard_ws['B6'] = "Satisfaction"
dashboard_ws['C6'] = '=TEXT(Data!B2,"0%")'  # Dynamic from data sheet
dashboard_ws['D6'] = '=TEXT(Data!D2,"+0%;-0%;0%")'

dashboard_ws['A8'] = "CX Initiatives"
dashboard_ws['B8'] = "Detail >"
dashboard_ws['B8'].font = Font(color="0070C0", italic=True)

# CX Initiative Performance
dashboard_ws['A10'] = "CX Initiative Performance"
dashboard_ws['A10'].font = Font(size=12, bold=True)

initiatives = [
    ("Onboarding Experience", '=Data!F3'),
    ("Support Response Time", '=Data!F4'),
    ("Self-Service Adoption", '=Data!F5'),
    ("Feedback Loop", '=Data!F6'),
    ("NPS Follow-up", '=Data!F7')
]

for i, (name, formula) in enumerate(initiatives, start=11):
    dashboard_ws[f'A{i}'] = name
    dashboard_ws[f'B{i}'] = formula

# Customer Satisfaction Section
dashboard_ws['E3'] = "Customer Satisfaction"
dashboard_ws['E3'].font = Font(bold=True)

dashboard_ws['F3'] = "Overall CX Performance"
dashboard_ws['F3'].font = Font(bold=True)

dashboard_ws['E4'] = "Complete ON TRACK"
dashboard_ws['F4'] = '=Data!B12&"% | "&Data!B13&" Weeks Left"'

dashboard_ws['E5'] = "Detail >"
dashboard_ws['E5'].font = Font(color="0070C0", italic=True)

# Journey Roadmap
dashboard_ws['A18'] = "Journey Roadmap"
dashboard_ws['A18'].font = Font(size=12, bold=True)

dashboard_ws['B19'] = "Pain Points Identified"
dashboard_ws['C19'] = '=Data!B7'
dashboard_ws['D19'] = '=IF(Data!D7>0,"+"&Data!D7,Data!D7)'

# Customer Journey Roadmap Timeline
dashboard_ws['A21'] = "Customer Journey Roadmap"
dashboard_ws['A21'].font = Font(bold=True)

months = ["August", "September", "October", "November", "December", "January", "February", "March"]
for i, month in enumerate(months, start=22):
    col_letter = get_column_letter(i-20)  # Starting from column B
    dashboard_ws[f'{col_letter}22'] = month

# Progress items
progress_items = [
    ("Journey Mapping Workshop", '=Data!B8&"% Complete"', '=Data!F8'),
    ("Voice of Customer Program", '=Data!B9&"% Complete"', '=Data!F9'),
    ("Touchpoint Optimization", '=Data!F10', '=Data!F10'),
    ("Friction Points Analysis", '=Data!F11', '=Data!F11')
]

for i, (name, progress, status) in enumerate(progress_items, start=24):
    dashboard_ws[f'A{i}'] = name
    dashboard_ws[f'B{i}'] = progress
    # Status will be shown via conditional formatting

# ==================== STYLING ====================
# Define colors
blue_fill = PatternFill(start_color="2E74B5", end_color="2E74B5", fill_type="solid")
light_blue_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
gray_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")

blue_font = Font(color="2E74B5")
white_font = Font(color="FFFFFF")
green_font = Font(color="006100")
red_font = Font(color="9C0006")
gray_font = Font(color="7F7F7F")

thin_border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)

# Style headers
for cell in ['A1', 'A3', 'A10', 'A18', 'A21']:
    if dashboard_ws[cell].value:
        dashboard_ws[cell].fill = light_blue_fill
        dashboard_ws[cell].border = thin_border

# Style section headers
for row in [5, 8, 19, 22]:
    for col in range(1, 9):
        cell = dashboard_ws.cell(row=row, column=col)
        if cell.value:
            cell.font = Font(bold=True)

# ==================== CONDITIONAL FORMATTING ====================
# Status-based formatting for Dashboard
for row in range(11, 16):  # Initiative status rows
    # Create conditional formatting based on cell value
    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"On Target"'], fill=green_fill, font=green_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Below Target"'], fill=yellow_fill, font=Font()))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Late"'], fill=red_fill, font=red_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"In Progress"'],
                   fill=PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid"),
                   font=white_font))

# Format for progress items (rows 24-27)
for row in range(24, 28):
    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='contains', formula=['"Complete"'], fill=green_fill, font=green_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='contains', formula=['"Late"'], fill=red_fill, font=red_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='contains', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

# Format satisfaction change
dashboard_ws.conditional_formatting.add('D6',
    FormulaRule(formula=['D6>0'], font=green_font, fill=green_fill))

dashboard_ws.conditional_formatting.add('D6',
    FormulaRule(formula=['D6<0'], font=red_font, fill=red_fill))

# Format pain points change
dashboard_ws.conditional_formatting.add('D19',
    FormulaRule(formula=['D19>0'], font=red_font))  # More pain points is bad

# Data bars for percentages in Data sheet
for row in [2, 4, 5, 6, 7, 8, 9, 12]:  # Rows with percentage data
    cell = f'B{row}'
    if row == 12:  # CX Performance
        data_ws.conditional_formatting.add(cell,
            FormulaRule(formula=[f'AND({cell}>=0,{cell}<=100)'],
                       fill=PatternFill(start_color="63BE7B", end_color="63BE7B", fill_type="solid")))
    else:
        data_ws.conditional_formatting.add(cell,
            CellIsRule(operator='greaterThan', formula=['=E' + str(row)],
                      fill=green_fill, font=green_font))
        data_ws.conditional_formatting.add(cell,
            CellIsRule(operator='lessThan', formula=['=E' + str(row)],
                      fill=yellow_fill, font=Font()))

# Status formatting for Data sheet
for row in range(2, 21):
    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"On Target"'], fill=green_fill, font=green_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Below Target"'], fill=yellow_fill, font=Font()))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Late"'], fill=red_fill, font=red_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"In Progress"'],
                   fill=PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid"),
                   font=white_font))

# ==================== ADD DATA VALIDATION ====================
from openpyxl.worksheet.datavalidation import DataValidation

# Create data validation for Status column in Data sheet
status_options = ["On Target", "Below Target", "Late", "Not Started", "In Progress"]
dv = DataValidation(type="list", formula1=f'"{",".join(status_options)}"', allow_blank=True)
dv.add('F2:F100')  # Apply to Status column rows
data_ws.add_data_validation(dv)

# ==================== ADD SPARKLINES (Simulated) ====================
# Add trend indicators using text
dashboard_ws['E6'] = "This year"
dashboard_ws['F6'] = "Last year"
dashboard_ws['E7'] = "◼◼◼◼◼◼◼◼◼◼"  # Simulated sparkline
dashboard_ws['F7'] = "◼◼◼◼◼◼◼◼◻◻"  # Simulated sparkline
dashboard_ws['E7'].font = Font(color="5B9BD5")
dashboard_ws['F7'].font = Font(color="ED7D31")

# ==================== FINAL TOUCHES ====================
# Add borders to dashboard
for row in range(1, 30):
    for col in range(1, 9):
        cell = dashboard_ws.cell(row=row, column=col)
        if cell.value:
            cell.border = thin_border
            cell.alignment = Alignment(horizontal='left', vertical='center')

# Center align specific cells
for cell in ['C6', 'D6', 'C19', 'D19', 'F4']:
    dashboard_ws[cell].alignment = Alignment(horizontal='center')

# Bold important values
dashboard_ws['C6'].font = Font(bold=True, size=11)
dashboard_ws['C19'].font = Font(bold=True, size=11)
dashboard_ws['F4'].font = Font(bold=True, size=11)

# Add timestamp
dashboard_ws['H30'] = f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
dashboard_ws['H30'].font = Font(size=8, italic=True, color="666666")

# ==================== SAVE THE FILE ====================
# Save the workbook
filename = "Customer_Experience_Dashboard.xlsx"
wb.save(filename)

print(f"✅ Excel dashboard created successfully!")
print(f"📂 File saved as: {filename}")
print(f"📊 Sheets: Dashboard, Data, Calculations")
print("\n📋 INSTRUCTIONS:")
print("1. Open 'Data' sheet to input your metrics")
print("2. Dashboard updates automatically")
print("3. Use dropdowns in 'Status' column (Column F)")
print("4. All formatting is conditional based on values")
print("\n🎯 Key Features:")
print("• Real-time updates between sheets")
print("• Color-coded status indicators")
print("• Data validation for consistent inputs")
print("• Support conversation metrics included")
print("• Professional dashboard layout")

# Show file size
file_size = os.path.getsize(filename)
print(f"\n💾 File Size: {file_size/1024:.1f} KB")

ValueError: Value must be one of {'between', 'notBetween', 'notEqual', 'containsText', 'notContains', 'beginsWith', 'greaterThanOrEqual', 'lessThan', 'greaterThan', 'equal', 'lessThanOrEqual', 'endsWith'}

In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment, NamedStyle
from openpyxl.formatting.rule import CellIsRule, FormulaRule
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.utils import get_column_letter
from datetime import datetime
import os

# Create a new workbook
wb = Workbook()

# Remove default sheet
default_sheet = wb.active
wb.remove(default_sheet)

# Create sheets
dashboard_ws = wb.create_sheet(title="Dashboard")
data_ws = wb.create_sheet(title="Data")
calculations_ws = wb.create_sheet(title="Calculations")

# Hide calculations sheet
calculations_ws.sheet_state = 'hidden'

# ==================== DATA SHEET ====================
# Set column widths for Data sheet
data_ws.column_dimensions['A'].width = 30
data_ws.column_dimensions['B'].width = 15
data_ws.column_dimensions['C'].width = 15
data_ws.column_dimensions['D'].width = 15
data_ws.column_dimensions['E'].width = 15
data_ws.column_dimensions['F'].width = 15

# Headers for Data sheet
data_headers = ["Metric", "Current Value", "Previous Value", "Change", "Target", "Status"]
data_ws.append(data_headers)

# Data rows
data_rows = [
    ["Overall Satisfaction", 0.74, 0.69, "", 0.75, "Below Target"],
    ["Onboarding Experience Score", 68, 65, "", 75, "Below Target"],
    ["Support Response Time (hrs)", 8, 7.5, "", 6, "Below Target"],
    ["Self-Service Adoption %", 35, 32, "", 45, "Below Target"],
    ["Feedback Loop Completion %", 92, 90, "", 90, "On Target"],
    ["NPS Follow-up Rate %", 88, 85, "", 85, "On Target"],
    ["Pain Points Identified", 18, 15, "", "", ""],
    ["Journey Mapping Workshop %", 87, "", "", 100, "In Progress"],
    ["Voice of Customer Program %", 30, "", "", 100, "In Progress"],
    ["Touchpoint Optimization", "", "", "", "", "Late"],
    ["Friction Points Analysis", "", "", "", "", "Not Started"],
    ["CX Performance Progress %", 47, 40, "", 100, "On Track"],
    ["Weeks Remaining", 12, "", "", "", ""],
    # Additional Support Metrics
    ["Support CSAT %", 85, 82, "", 90, "Below Target"],
    ["First Response Time (min)", 15, 18, "", 10, "Below Target"],
    ["Resolution Rate %", 92, 90, "", 95, "Below Target"],
    ["Escalation Rate %", 8, 10, "", 5, "Below Target"],
    ["Self-Service Deflection %", 25, 22, "", 35, "Below Target"],
    ["Customer Effort Score (1-7)", 2.5, 2.8, "", 2.0, "Below Target"],
    ["Sentiment Score (0-100)", 78, 75, "", 85, "Below Target"]
]

for row in data_rows:
    data_ws.append(row)

# Add formulas for Change column (row 2 to 6 for percentages, row 7 for count)
for row in range(2, 7):  # Rows 2-6
    data_ws[f'D{row}'] = f'=B{row}-C{row}'

# Pain points change
data_ws['D7'] = '=B7-C7'

# ==================== CALCULATIONS SHEET ====================
calculations_ws['A1'] = "CALCULATION SHEET - DO NOT EDIT"
calculations_ws['A2'] = "Automatic Calculations"
calculations_ws['B2'] = "Formulas"

# Add some useful formulas
calculations_ws['A3'] = "Satisfaction % Display"
calculations_ws['B3'] = '=TEXT(Data!B2,"0%")'

calculations_ws['A4'] = "Satisfaction Change Display"
calculations_ws['B4'] = '=TEXT(Data!D2,"+0%;-0%;0%")'

calculations_ws['A5'] = "Pain Points Change"
calculations_ws['B5'] = '=IF(Data!D7>0,"+"&Data!D7,Data!D7)'

calculations_ws['A6'] = "Progress Bar"
calculations_ws['B6'] = '=REPT("█",ROUND(Data!B12/10,0))'

# ==================== DASHBOARD SHEET ====================
# Set column widths
for col in range(1, 9):
    dashboard_ws.column_dimensions[get_column_letter(col)].width = 12

# Merge cells for header
dashboard_ws.merge_cells('A1:H1')
dashboard_ws['A1'] = "CUSTOMER EXPERIENCE DASHBOARD"
dashboard_ws['A1'].font = Font(size=16, bold=True, color="2E74B5")
dashboard_ws['A1'].alignment = Alignment(horizontal='center')

# GENERAL Section
dashboard_ws['A3'] = "GENERAL"
dashboard_ws['A3'].font = Font(size=12, bold=True)

# Overview
dashboard_ws['A5'] = "Overview"
dashboard_ws['A5'].font = Font(bold=True)

dashboard_ws['B6'] = "Satisfaction"
dashboard_ws['C6'] = '=TEXT(Data!B2,"0%")'  # Dynamic from data sheet
dashboard_ws['D6'] = '=TEXT(Data!D2,"+0%;-0%;0%")'

dashboard_ws['A8'] = "CX Initiatives"
dashboard_ws['B8'] = "Detail >"
dashboard_ws['B8'].font = Font(color="0070C0", italic=True)

# CX Initiative Performance
dashboard_ws['A10'] = "CX Initiative Performance"
dashboard_ws['A10'].font = Font(size=12, bold=True)

initiatives = [
    ("Onboarding Experience", '=Data!F3'),
    ("Support Response Time", '=Data!F4'),
    ("Self-Service Adoption", '=Data!F5'),
    ("Feedback Loop", '=Data!F6'),
    ("NPS Follow-up", '=Data!F7')
]

for i, (name, formula) in enumerate(initiatives, start=11):
    dashboard_ws[f'A{i}'] = name
    dashboard_ws[f'B{i}'] = formula

# Customer Satisfaction Section
dashboard_ws['E3'] = "Customer Satisfaction"
dashboard_ws['E3'].font = Font(bold=True)

dashboard_ws['F3'] = "Overall CX Performance"
dashboard_ws['F3'].font = Font(bold=True)

dashboard_ws['E4'] = "Complete ON TRACK"
dashboard_ws['F4'] = '=Data!B12&"% | "&Data!B13&" Weeks Left"'

dashboard_ws['E5'] = "Detail >"
dashboard_ws['E5'].font = Font(color="0070C0", italic=True)

# Journey Roadmap
dashboard_ws['A18'] = "Journey Roadmap"
dashboard_ws['A18'].font = Font(size=12, bold=True)

dashboard_ws['B19'] = "Pain Points Identified"
dashboard_ws['C19'] = '=Data!B7'
dashboard_ws['D19'] = '=IF(Data!D7>0,"+"&Data!D7,Data!D7)'

# Customer Journey Roadmap Timeline
dashboard_ws['A21'] = "Customer Journey Roadmap"
dashboard_ws['A21'].font = Font(bold=True)

months = ["August", "September", "October", "November", "December", "January", "February", "March"]
for i, month in enumerate(months, start=22):
    col_letter = get_column_letter(i-20)  # Starting from column B
    dashboard_ws[f'{col_letter}22'] = month

# Progress items
progress_items = [
    ("Journey Mapping Workshop", '=Data!B8&"% Complete"', '=Data!F8'),
    ("Voice of Customer Program", '=Data!B9&"% Complete"', '=Data!F9'),
    ("Touchpoint Optimization", '=Data!F10', '=Data!F10'),
    ("Friction Points Analysis", '=Data!F11', '=Data!F11')
]

for i, (name, progress, status) in enumerate(progress_items, start=24):
    dashboard_ws[f'A{i}'] = name
    dashboard_ws[f'B{i}'] = progress
    # Status will be shown via conditional formatting

# ==================== STYLING ====================
# Define colors
blue_fill = PatternFill(start_color="2E74B5", end_color="2E74B5", fill_type="solid")
light_blue_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
gray_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")

blue_font = Font(color="2E74B5")
white_font = Font(color="FFFFFF")
green_font = Font(color="006100")
red_font = Font(color="9C0006")
gray_font = Font(color="7F7F7F")

thin_border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)

# Style headers
for cell in ['A1', 'A3', 'A10', 'A18', 'A21']:
    if dashboard_ws[cell].value:
        dashboard_ws[cell].fill = light_blue_fill
        dashboard_ws[cell].border = thin_border

# Style section headers
for row in [5, 8, 19, 22]:
    for col in range(1, 9):
        cell = dashboard_ws.cell(row=row, column=col)
        if cell.value:
            cell.font = Font(bold=True)

# ==================== CONDITIONAL FORMATTING ====================
# Status-based formatting for Dashboard
for row in range(11, 16):  # Initiative status rows
    # Create conditional formatting based on cell value
    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"On Target"'], fill=green_fill, font=green_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Below Target"'], fill=yellow_fill, font=Font()))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Late"'], fill=red_fill, font=red_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"In Progress"'],
                   fill=PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid"),
                   font=white_font))

# FIXED: Use 'containsText' instead of 'contains' for progress items
for row in range(24, 28):
    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='containsText', formula=['"Complete"'], fill=green_fill, font=green_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Late"'], fill=red_fill, font=red_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

    dashboard_ws.conditional_formatting.add(f'B{row}',
        CellIsRule(operator='equal', formula=['"In Progress"'],
                   fill=PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid"),
                   font=white_font))

# Format satisfaction change
dashboard_ws.conditional_formatting.add('D6',
    FormulaRule(formula=['D6>0'], font=green_font, fill=green_fill))

dashboard_ws.conditional_formatting.add('D6',
    FormulaRule(formula=['D6<0'], font=red_font, fill=red_fill))

# Format pain points change
dashboard_ws.conditional_formatting.add('D19',
    FormulaRule(formula=['D19>0'], font=red_font))  # More pain points is bad

# Data bars for percentages in Data sheet
for row in [2, 4, 5, 6, 7, 8, 9, 12]:  # Rows with percentage data
    cell = f'B{row}'
    if row == 12:  # CX Performance
        data_ws.conditional_formatting.add(cell,
            FormulaRule(formula=[f'AND({cell}>=0,{cell}<=100)'],
                       fill=PatternFill(start_color="63BE7B", end_color="63BE7B", fill_type="solid")))
    else:
        data_ws.conditional_formatting.add(cell,
            CellIsRule(operator='greaterThan', formula=['=E' + str(row)],
                      fill=green_fill, font=green_font))
        data_ws.conditional_formatting.add(cell,
            CellIsRule(operator='lessThan', formula=['=E' + str(row)],
                      fill=yellow_fill, font=Font()))

# Status formatting for Data sheet
for row in range(2, 21):
    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"On Target"'], fill=green_fill, font=green_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Below Target"'], fill=yellow_fill, font=Font()))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Late"'], fill=red_fill, font=red_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"Not Started"'], fill=gray_fill, font=gray_font))

    data_ws.conditional_formatting.add(f'F{row}',
        CellIsRule(operator='equal', formula=['"In Progress"'],
                   fill=PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid"),
                   font=white_font))

# ==================== ADD DATA VALIDATION ====================
from openpyxl.worksheet.datavalidation import DataValidation

# Create data validation for Status column in Data sheet
status_options = ["On Target", "Below Target", "Late", "Not Started", "In Progress"]
dv = DataValidation(type="list", formula1=f'"{",".join(status_options)}"', allow_blank=True)
dv.add('F2:F100')  # Apply to Status column rows
data_ws.add_data_validation(dv)

# ==================== ADD SPARKLINES (Simulated) ====================
# Add trend indicators using text
dashboard_ws['E6'] = "This year"
dashboard_ws['F6'] = "Last year"
dashboard_ws['E7'] = "◼◼◼◼◼◼◼◼◼◼"  # Simulated sparkline
dashboard_ws['F7'] = "◼◼◼◼◼◼◼◼◻◻"  # Simulated sparkline
dashboard_ws['E7'].font = Font(color="5B9BD5")
dashboard_ws['F7'].font = Font(color="ED7D31")

# ==================== FINAL TOUCHES ====================
# Add borders to dashboard
for row in range(1, 30):
    for col in range(1, 9):
        cell = dashboard_ws.cell(row=row, column=col)
        if cell.value:
            cell.border = thin_border
            cell.alignment = Alignment(horizontal='left', vertical='center')

# Center align specific cells
for cell in ['C6', 'D6', 'C19', 'D19', 'F4']:
    dashboard_ws[cell].alignment = Alignment(horizontal='center')

# Bold important values
dashboard_ws['C6'].font = Font(bold=True, size=11)
dashboard_ws['C19'].font = Font(bold=True, size=11)
dashboard_ws['F4'].font = Font(bold=True, size=11)

# Add timestamp
dashboard_ws['H30'] = f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
dashboard_ws['H30'].font = Font(size=8, italic=True, color="666666")

# ==================== SAVE THE FILE ====================
# Save the workbook
filename = "Customer_Experience_Dashboard.xlsx"
wb.save(filename)

print(f"✅ Excel dashboard created successfully!")
print(f"📂 File saved as: {filename}")
print(f"📊 Sheets: Dashboard, Data, Calculations")
print("\n📋 INSTRUCTIONS:")
print("1. Open 'Data' sheet to input your metrics")
print("2. Dashboard updates automatically")
print("3. Use dropdowns in 'Status' column (Column F)")
print("4. All formatting is conditional based on values")
print("\n🎯 Key Features:")
print("• Real-time updates between sheets")
print("• Color-coded status indicators")
print("• Data validation for consistent inputs")
print("• Support conversation metrics included")
print("• Professional dashboard layout")

# Show file size
file_size = os.path.getsize(filename)
print(f"\n💾 File Size: {file_size/1024:.1f} KB")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix,
accuracy_score
# Machine Learning models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# Ensemble models
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix,
accuracy_score
# Machine Learning models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# Ensemble models
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier